# Walmart Store Sales - Multi-Method Forecasting Lab\nStore 1. The final 12 weeks are held out before fitting any forecasting method.

In [ ]:
import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\n\ndf = pd.read_csv("Walmart_Store_sales.csv")\ndf["Date"] = pd.to_datetime(df["Date"], format="%d-%m-%Y")\n\nSTORE_ID = 1\nstore_df = df[df["Store"] == STORE_ID].sort_values("Date").reset_index(drop=True)\n\ntrain_df = store_df.iloc[:-12].copy()\ntest_df = store_df.iloc[-12:].copy()\n\n# 1) 4-week Simple Moving Average\nsma_forecast = np.repeat(train_df["Weekly_Sales"].tail(4).mean(), len(test_df))\n\n# 2) Simple Exponential Smoothing, coded directly\ndef ses_level(series, alpha):\n    level = series.iloc[0]\n    for y in series.iloc[1:]:\n        level = alpha*y + (1-alpha)*level\n    return level\n\nses03_forecast = np.repeat(ses_level(train_df["Weekly_Sales"], 0.3), len(test_df))\nses06_forecast = np.repeat(ses_level(train_df["Weekly_Sales"], 0.6), len(test_df))\n\n# 3) Seasonal naive with Holiday_Flag\n# For each test week, look 52 weeks back and use the closest TRAINING week\n# with the same Holiday_Flag. No test sales are used for fitting.\nseasonal_forecast = []\nfor _, row in test_df.iterrows():\n    target_date = row["Date"] - pd.Timedelta(weeks=52)\n    candidates = train_df[train_df["Holiday_Flag"] == row["Holiday_Flag"]].copy()\n    nearest_idx = (candidates["Date"] - target_date).abs().idxmin()\n    seasonal_forecast.append(train_df.loc[nearest_idx, "Weekly_Sales"])\nseasonal_forecast = np.array(seasonal_forecast)\n\ndef mape(actual, forecast):\n    return np.mean(np.abs((actual - forecast) / actual)) * 100\n\nactual = test_df["Weekly_Sales"].to_numpy()\ncomparison = pd.DataFrame({\n    "Method": ["4-week SMA", "SES alpha=0.3", "SES alpha=0.6", "Seasonal Naive"],\n    "MAPE (%)": [\n        mape(actual, sma_forecast),\n        mape(actual, ses03_forecast),\n        mape(actual, ses06_forecast),\n        mape(actual, seasonal_forecast)\n    ]\n}).sort_values("MAPE (%)")\n\nprint(comparison.to_string(index=False))\n\nplt.figure(figsize=(11,6))\nplt.plot(test_df["Date"], actual, marker="o", label="Actual")\nplt.plot(test_df["Date"], sma_forecast, marker="o", label="4-week SMA")\nplt.plot(test_df["Date"], ses03_forecast, marker="o", label="SES 0.3")\nplt.plot(test_df["Date"], ses06_forecast, marker="o", label="SES 0.6")\nplt.plot(test_df["Date"], seasonal_forecast, marker="o", label="Seasonal Naive")\nplt.title(f"Walmart Store {STORE_ID}: 12-Week Forecast Comparison")\nplt.xlabel("Date")\nplt.ylabel("Weekly Sales")\nplt.xticks(rotation=45)\nplt.legend()\nplt.tight_layout()\nplt.show()\n